# Tools to compute manufactured solutions

Computing manufactured solutions to test the implementation
of numerical methods can be errorprone even for simple problems
such as Poisson problem. Some forgotten constants, signs etc.
can create quite some headaches when you're debugging your code
just to realize after hours of bug hunting that in fact your
test example had an error to begin with.

For more complicated examples such as the Laplace-Beltrami operator 
on surfaces, or highly non-linear fluid-structure interaction problems,
it is (at least for me) impossible to calculate a nontrivial manufactured 
solutions **quickly and correctly**. 

This notebook illustrates how to use the Python module [Sympy](https://docs.sympy.org/latest/index.html) to symbolically compute the result of various differential operators to 
given analytical manufactured solutions. And while many differential operators are
already directly available from Sympy, we also provide
short-cut functions for 
some less common differential operators related to surface PDES.
such as surface gradients, divergence, Laplace-Beltrami operator etc.


### Some useful functions

We define some differential operators which are missing in sympy.

In [4]:
import sympy as sp 
from sympy.vector import CoordSys3D
from sympy.printing import latex, pretty
from IPython.display import display, Math

In [5]:
# Define modified print-related function in case expressions get to large
# to be rendered in MathJax
# Count number of operations

def _count_ops(e):
    if hasattr(e, "shape"):
        return sum(_count_ops(a) for a in e)
    else:
        return e.count_ops()

def eprint(e, name=None, maxsize=500):
    
    """Pretty-print expression, using either LaTeX/MathJax for small
    expressions or SymPy unicode for large expressions, as determined
    by the variable maxsize."""

    if _count_ops(e) <= maxsize:
        s = latex(e)
        if name is not None: s = name + " = " + s
        display(Math(s))

    else:
        s = pretty(e)
        if name is not None: s = name + " = " + s
        display(s)

We now define the relevant PDE operators for our problems classes.
Beside the well-known operators such $\Delta$, $\nabla$ etc. we recall that
the Laplace-Beltrami operator $\Delta_{\Gamma} u$ can be calculated as follows:

$$ \Delta_{\Gamma} u = \Delta u - n_{\Gamma} \cdot \nabla^2 u n_{\Gamma} - \textrm{tr}(\nabla n_{\Gamma}) \nabla u \cdot n_{\Gamma},
$$

where $n_{\Gamma}$ denotes the surface normal.

After we haved defined all the necessary operators, we can simply pick our favorite colorful solution
$u$ and compute corresponding data for the PDE problem at hand. For instance for the **Poisson problem**
$$
\begin{align}
 -\Delta u &= f \quad \text{in } \Omega
 \\
 u &= g \quad \text{on } \partial \Omega
\end{align}
$$
we can compute $f$ directly from our chose $u$.

To manufacture a solution for the **Laplace Beltrami problem**,
we also need to provide a level-set description $M = \{ \phi = 0 \}$ of our favorite manifold $M$ to compute the normal-field $n_{\Gamma}$ by 
$$n_{\Gamma} = \dfrac{\nabla\phi}{|\nabla\phi|},$$ 
as it is part of the definition of $\Delta_{\Gamma}$. Here are the corresponding Python snippets.

In [4]:
# Use standard Cartesian coordinates
x,y,z = symbols("x y z")
xx = Matrix([x, y, z])

levelset_str = "x**2+y**2 - 1"

phi = eval(levelset_str)
print("Got the level set function: ", phi)
eprint(phi)

Got the level set function:  x**2 + y**2 - 1


<IPython.core.display.Math object>

In [110]:
def vec_norm(xx):
    return sum([xi*xi for xi in xx])**0.5

def compute_normal(phi, xx):
    return Grad(phi, xx)/vec_norm(Grad(phi, xx))

# Define various differential operators which SymPy does not provide directly.
# TODO: Check whether this is still true!

def Grad(f, x):
    "Compute gradient."
    if not hasattr(f, "shape"):
        return sp.Matrix([sp.diff(f, xj) for xj in x])
    elif len(f.shape) == 2 and f.shape[1] == 1:
        return sp.Matrix([[sp.diff(fi, xj) for xj in x] for fi in f])
    else:
        raise RuntimeError("Don't know how to take grad.")

def Div(f, x):
    "Compute divergence."
    if len(f.shape) == 2 and f.shape[1] == 1:
        m = f.shape[0]
        return sum([sp.diff(f[i], x[i]) for i in range(m)])

    elif len(f.shape) == 2:
        m, n = f.shape
        return sp.Matrix([sum([sp.diff(f[i, j], x[j]) for j in range(n)])
                       for i in range(m)])
    else:
        raise RuntimeError("Don't know how to take div.")

def Laplace(f, x):
    "Compute the (full) Laplacian"
    return Div(Grad(f,x),x)
    
def Hess(f, x):
    "Compute the Hessian of f"
    return Grad(Grad(f, X), X)

def Tr(f):
    "Compute trace."
    if len(f.shape) == 2 and f.shape[0] == f.shape[1]:
        m = f.shape[0]
        return sum([f[i,i] for i in range(m)])
    else:
        raise RuntimeError("Don't know how to take traces.")

def P(n):
    "Compute the projection into tangent component matrix."
    return sp.eye(n.shape[0]) - sp.Matrix([[sp.simplify(ni*nj) for nj in n] for ni in n])

def Q(n):
    "Compute the projection into normal component matrix."
    return sp.eye(n.shape[0]) - P(n)

def SurfGrad(f, x, n):
    return P(n)*Grad(f,x)
    
def SurfDiv(f, x, n):
    return Div(f, x) - (Grad(f, x)*n).dot(n)

# def SurfLaplace(f,x,n):
    
def LaplaceBeltrami(f,x,n):
    "Compute Laplace-Beltrami operator of u"
    return Laplace(f,x) - n.dot(Hess(f,x).dot(n)) - Tr(Grad(n,x))*Grad(f,x).dot(n)

Next, we define a little wrapper function ```get_solution_str``` which for a given surface (defined by a level set)
and an analytical reference solution will compute the right-hand side $f$ to the problem
$$
-\epsilon\Delta_{\Gamma} u + c u = f \quad \text{on } \Gamma.
$$
It will also compute the tangential gradient of the manufactured solution.

In [95]:
x, y, z, t = sp.symbols("x y z t")
X = sp.Matrix([x, y, z]) 

phi = (x*x + y*y + z*z)**0.5 - 1
eprint(phi, name="\\phi")

grad_phi = Grad(phi, X)
eprint(grad_phi, name="\\nabla\\phi")

normal = sp.simplify(compute_normal(phi, X))
eprint(normal, name="\\boldsymbol{n}_{\\Gamma}") 

hess_phi = Hess(phi, X)
eprint(hess_phi, name="D^2\\phi")

hess_phi_2= Grad(normal, X)
eprint(hess_phi_2, name="D^2\\phi")

laplace_phi = Laplace(phi, X)
eprint(laplace_phi, name="\\Delta\\phi")


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [78]:
# l = laplace_phi.subs(-1.0, "1").subs(x*x+y*y+z*z, 1)
# print(l.subs(x**2 + y**2 + z**2, 1))



x**2 + y**2 + z**2 + 3.0


In [96]:
u = sp.Matrix([x*x+x*x, x*y*y+ 2*x*y*y, x*z*z])
eprint(u, name="\\boldsymbol{u}")

grad_u = Grad(u, X)
eprint(grad_u, name="\\nabla\\boldsymbol{u}")

div_u = Div(u, X)
eprint(div_u, name="\\nabla\\cdot\\boldsymbol{u}")

laplace_u = Laplace(u, X)
eprint(laplace_u, name="\\Delta u")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

(3, 3)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

## A little example

Let's compute the $\Delta_{\Gamma} u + u$ for $\Gamma = S^2 = \{\phi = 0\}$ where
$$
\begin{cases}
u(x,y,z) = \sin(\dfrac{\pi x}{2}) \sin(\dfrac{\pi y}{2}) \sin(\dfrac{\pi z}{2})
\\
\phi(x,y,z) = x^2 + y^2 + z^2 - 1
\end{cases}
$$

In [112]:
u_str = "sin(x/2)*sin(y/2)*sin(z/2)"
u = sp.sympify(u_str)
eprint(u, name="u")

grad_u = Grad(u, X)
eprint(grad_u, name="\\nabla u")

hess_u = Hess(u, X)
eprint(hess_u, name="D^2u")

laplace_u = Laplace(u, X)
eprint(laplace_u, name="\\Delta u")

phi_str = "x**2 + y**2 + z**2 - 1"
phi =  sp.sympify(phi_str)

normal = sp.simplify(compute_normal(phi, X))
eprint(normal, name="\\boldsymbol{n}_{\\Gamma}") 

surfgrad_u = SurfGrad(u, X, normal).subs(x*x+y*y+z*z, 1)
eprint(surfgrad_u)

# surfgrad_u = surfgrad_u.subs(x*x+y*y+z*z, 1)
# normal = compute_normal(u_str)
# f, tang_grad_u = get_solution_str(u_str, levelset_str)

# eprint(f, name="f")
# eprint(tang_grad_u, name="\\nabla_{\\Gamma}u")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [ ]:

normal = compute_normal(phi, xx)
print('Normal:', normal)

u = parse_expr(u_str)
b = Matrix([parse_expr(bb) for bb in b_str])
print("b: ", b)
c = parse_expr(c_str)
print("c: ", c)
epsilon = parse_expr(epsilon_str)
print("epsilon: ", epsilon)
# f = simplify(-diff_coeff*LaplaceBeltrami(u, xx, normal) + reac_coeff*u)
f = -epsilon*LaplaceBeltrami(u, xx, normal) \
    + c*u \
    + b.dot(Project_Grad(u, xx, normal))
print('RHS f: ', f)

tang_grad_u = Project_Grad(u, xx, normal)


In [ ]:
def get_solution_str(u_str, levelset_str, epsilon_str="1", b_str=["0","0","0"], c_str="1"):
    # Use standard Cartesian coordinates
    x,y,z = symbols("x y z")
    xx = Matrix([x, y, z])

    phi = eval(levelset_str)
    print("Got the level set function: ", phi)
    normal = compute_normal(phi, xx)
    print('Normal:', normal)
    
    u = parse_expr(u_str)
    b = Matrix([parse_expr(bb) for bb in b_str])
    print("b: ", b)
    c = parse_expr(c_str)
    print("c: ", c)
    epsilon = parse_expr(epsilon_str)
    print("epsilon: ", epsilon)
    # f = simplify(-diff_coeff*LaplaceBeltrami(u, xx, normal) + reac_coeff*u)
    f = -epsilon*LaplaceBeltrami(u, xx, normal) \
        + c*u \
        + b.dot(Project_Grad(u, xx, normal))
    print('RHS f: ', f)
    
    tang_grad_u = Project_Grad(u, xx, normal)
    return f, tang_grad_u
    #return str(f), [str(D) for D in project_grad]
